# 🔧 Notebook 07 — Optimización de Hiperparámetros (HPO)
## Tesis: Asistente Conversacional Inteligente para iTimeControl

**Objetivo:** Encontrar la configuración de hiperparámetros óptima mediante búsqueda sistemática (Random Search y Bayesian Optimization), con presupuesto controlado y evidencia reproducible.

---
### Puntos cubiertos en este notebook:

| Punto | Descripción |
|---|---|
| **1. Experimentos comparables** | Random Search vs Bayesian Optimization con pruning/early stopping |
| **2. Logs + artefactos** | Cada trial registrado; artefactos (modelos, métricas) guardados |
| **3. Tabla top-k y gráfico de evolución** | Top-10 configs y curva de mejor métrica por trial |
| **4. Resumen y decisión** | Espacio de búsqueda, presupuesto y config ganadora |

---
## 1. Setup e importaciones

In [ ]:
# Instalar optuna si no está disponible
try:
    import optuna
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'])
    import optuna

import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
import pickle

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
COLORS = ['#4C72B0', '#55A868', '#C44E52', '#DD8452', '#8172B2', '#937860']

ROOT     = Path('..')
DATASETS = ROOT / 'data' / 'datasets'
LOGS_DIR = ROOT / 'logs'
ARTS_DIR = ROOT / 'artifacts'
LOGS_DIR.mkdir(exist_ok=True)
ARTS_DIR.mkdir(exist_ok=True)

print(f'✅ Setup completado | Optuna {optuna.__version__}')

In [ ]:
# Carga de datos y etiquetado de intención (igual que notebooks anteriores)
def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding='utf-8').splitlines() if l.strip()]

train = load_jsonl(DATASETS / 'train.jsonl')
val   = load_jsonl(DATASETS / 'val.jsonl')
test  = load_jsonl(DATASETS / 'test.jsonl')

all_questions = [r['instruction'] for r in train + val + test]

INTENT_KW = {
    'registro_asistencia': ['registrar','marcar','asistencia','entrada','salida','marcacion'],
    'reportes':            ['reporte','informe','exportar','excel','estadistica','descargar'],
    'horarios':            ['horario','turno','jornada','calendario','tolerancia'],
    'empleados':           ['empleado','personal','nuevo','agregar'],
    'solicitudes':         ['permiso','vacacion','ausencia','justificar','solicitud'],
    'configuracion':       ['configurar','backup','rol','dispositivo','feriado','contrasena'],
}

def assign_intent(text):
    tl = text.lower(); best, sc = 'general', 0
    for intent, kws in INTENT_KW.items():
        s = sum(1 for kw in kws if kw in tl)
        if s > sc: best, sc = intent, s
    return best

labels = np.array([assign_intent(q) for q in all_questions])
print(f'Total muestras: {len(all_questions)}')
print('Distribución:', dict(Counter(labels)))

---
## 2. Experimentos comparables: Random Search vs Bayesian Optimization
### con pruning / early stopping

Se comparan dos estrategias de búsqueda bajo el **mismo espacio de hiperparámetros** y el **mismo presupuesto de trials** para que los resultados sean comparables.

- **Random Search** — muestrea el espacio uniformemente al azar.
- **Bayesian Optimization (TPE)** — usa los trials anteriores para dirigir la búsqueda hacia regiones prometedoras.
- **Pruning / early stopping** — Optuna elimina trials que no superan el mejor valor parcial (MedianPruner), ahorrando cómputo.

In [ ]:
# Espacio de hiperparámetros compartido
SEARCH_SPACE = {
    'max_features':  (50, 300),          # TF-IDF
    'ngram_max':     [1, 2, 3],          # (1,1), (1,2), (1,3)
    'sublinear_tf':  [True, False],
    'n_estimators':  (50, 300),          # Random Forest
    'max_depth':     (3, 15),
    'min_samples_leaf': (1, 10),
}

N_TRIALS   = 40   # presupuesto total por estrategia
CV_SPLITS  = 5

skf = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=42)

def objective(trial):
    """Función objetivo compartida — usada por ambas estrategias."""
    max_feat   = trial.suggest_int('max_features', *SEARCH_SPACE['max_features'])
    ngram_max  = trial.suggest_categorical('ngram_max', SEARCH_SPACE['ngram_max'])
    sublin     = trial.suggest_categorical('sublinear_tf', SEARCH_SPACE['sublinear_tf'])
    n_est      = trial.suggest_int('n_estimators', *SEARCH_SPACE['n_estimators'])
    depth      = trial.suggest_int('max_depth', *SEARCH_SPACE['max_depth'])
    min_leaf   = trial.suggest_int('min_samples_leaf', *SEARCH_SPACE['min_samples_leaf'])

    f1_scores = []
    for step, (tr_idx, te_idx) in enumerate(skf.split(all_questions, labels)):
        X_tr = [all_questions[i] for i in tr_idx]
        X_te = [all_questions[i] for i in te_idx]
        y_tr, y_te = labels[tr_idx], labels[te_idx]

        vec = TfidfVectorizer(
            max_features=max_feat,
            ngram_range=(1, ngram_max),
            strip_accents='unicode',
            sublinear_tf=sublin,
        )
        Xt = vec.fit_transform(X_tr)
        Xv = vec.transform(X_te)

        clf = RandomForestClassifier(
            n_estimators=n_est, max_depth=depth,
            min_samples_leaf=min_leaf, random_state=42, n_jobs=-1
        )
        clf.fit(Xt, y_tr)
        f1 = f1_score(y_te, clf.predict(Xv), average='weighted', zero_division=0)
        f1_scores.append(f1)

        # Pruning: reportar valor parcial tras cada fold
        trial.report(np.mean(f1_scores), step)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return float(np.mean(f1_scores))

print(f'Espacio definido | presupuesto: {N_TRIALS} trials × 2 estrategias = {N_TRIALS*2} trials totales')

In [ ]:
# ── Random Search ────────────────────────────────────────────────────────────
print('🔀 Ejecutando Random Search...')
study_random = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.RandomSampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
    study_name='random_search',
)
study_random.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

print(f'✅ Random Search finalizado')
print(f'   Mejor F1: {study_random.best_value:.4f}')
print(f'   Trials completados: {len([t for t in study_random.trials if t.state == optuna.trial.TrialState.COMPLETE])}')
print(f'   Trials podados:     {len([t for t in study_random.trials if t.state == optuna.trial.TrialState.PRUNED])}')

In [ ]:
# ── Bayesian Optimization (TPE) ──────────────────────────────────────────────
print('🧠 Ejecutando Bayesian Optimization (TPE)...')
study_bayes = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
    study_name='bayesian_tpe',
)
study_bayes.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

print(f'✅ Bayesian Optimization finalizado')
print(f'   Mejor F1: {study_bayes.best_value:.4f}')
print(f'   Trials completados: {len([t for t in study_bayes.trials if t.state == optuna.trial.TrialState.COMPLETE])}')
print(f'   Trials podados:     {len([t for t in study_bayes.trials if t.state == optuna.trial.TrialState.PRUNED])}')

---
## 3. Logs + artefactos guardados

Todos los trials (parámetros, métricas, estado) se persisten en `logs/` como JSON y CSV. El modelo ganador se serializa en `artifacts/`.

In [ ]:
def trials_to_records(study, strategy_name):
    records = []
    for t in study.trials:
        records.append({
            'strategy':  strategy_name,
            'trial_num': t.number,
            'state':     t.state.name,
            'f1_score':  round(t.value, 6) if t.value is not None else None,
            **{f'param_{k}': v for k, v in t.params.items()},
            'duration_s': round(t.duration.total_seconds(), 2) if t.duration else None,
        })
    return records

records_random = trials_to_records(study_random, 'RandomSearch')
records_bayes  = trials_to_records(study_bayes,  'BayesianTPE')
all_records    = records_random + records_bayes

# CSV con todos los trials
df_log = pd.DataFrame(all_records)
df_log.to_csv(LOGS_DIR / 'hpo_all_trials.csv', index=False)
print(f'✅ CSV guardado: logs/hpo_all_trials.csv  ({len(df_log)} filas)')

# JSON detallado por estrategia
hpo_log = {
    'fecha':        datetime.now().strftime('%Y-%m-%d %H:%M'),
    'n_trials_por_estrategia': N_TRIALS,
    'espacio':      {k: list(v) if isinstance(v, list) else str(v) for k, v in SEARCH_SPACE.items()},
    'RandomSearch': {
        'mejor_f1':    round(study_random.best_value, 6),
        'mejores_params': study_random.best_params,
        'completados': len([t for t in study_random.trials if t.state == optuna.trial.TrialState.COMPLETE]),
        'podados':     len([t for t in study_random.trials if t.state == optuna.trial.TrialState.PRUNED]),
    },
    'BayesianTPE': {
        'mejor_f1':    round(study_bayes.best_value, 6),
        'mejores_params': study_bayes.best_params,
        'completados': len([t for t in study_bayes.trials if t.state == optuna.trial.TrialState.COMPLETE]),
        'podados':     len([t for t in study_bayes.trials if t.state == optuna.trial.TrialState.PRUNED]),
    },
}
with open(LOGS_DIR / 'hpo_log.json', 'w', encoding='utf-8') as f:
    json.dump(hpo_log, f, indent=2, ensure_ascii=False)
print('✅ JSON guardado: logs/hpo_log.json')

# Artefacto: modelo ganador (mejor entre las dos estrategias)
winner_study = study_bayes if study_bayes.best_value >= study_random.best_value else study_random
winner_name  = 'BayesianTPE' if winner_study is study_bayes else 'RandomSearch'
bp           = winner_study.best_params

vec_best = TfidfVectorizer(
    max_features=bp['max_features'], ngram_range=(1, bp['ngram_max']),
    strip_accents='unicode', sublinear_tf=bp['sublinear_tf'],
)
X_best = vec_best.fit_transform(all_questions)
clf_best = RandomForestClassifier(
    n_estimators=bp['n_estimators'], max_depth=bp['max_depth'],
    min_samples_leaf=bp['min_samples_leaf'], random_state=42, n_jobs=-1
)
clf_best.fit(X_best, labels)

with open(ARTS_DIR / 'hpo_best_model.pkl', 'wb') as f:
    pickle.dump({'vectorizer': vec_best, 'classifier': clf_best, 'params': bp}, f)
print(f'✅ Artefacto guardado: artifacts/hpo_best_model.pkl  (estrategia: {winner_name})')

---
## 4. Tabla top-k y gráfico de evolución

- **Tabla top-10:** Las 10 mejores configuraciones de cada estrategia, ordenadas por F1.
- **Gráfico de evolución:** Cómo mejora el mejor F1 acumulado a medida que avanzan los trials.

In [ ]:
K = 10

# Top-k por estrategia
def top_k_df(records, k=10):
    df = pd.DataFrame(records)
    df = df[df['state'] == 'COMPLETE'].copy()
    df = df.sort_values('f1_score', ascending=False).head(k).reset_index(drop=True)
    df.index += 1
    return df

df_top_random = top_k_df(records_random, K)
df_top_bayes  = top_k_df(records_bayes,  K)

print(f'=== TOP-{K} — Random Search ===')
param_cols = [c for c in df_top_random.columns if c.startswith('param_')]
display_cols = ['trial_num', 'f1_score'] + param_cols
print(df_top_random[display_cols].to_string())

print(f'\n=== TOP-{K} — Bayesian TPE ===')
print(df_top_bayes[display_cols].to_string())

# Guardar CSV de top-k
df_top_random.to_csv(LOGS_DIR / 'hpo_top10_random.csv', index=True)
df_top_bayes.to_csv( LOGS_DIR / 'hpo_top10_bayes.csv',  index=True)
print('\n✅ Tablas top-k guardadas en logs/')

In [ ]:
def best_so_far(study):
    """Mejor F1 acumulado trial a trial (solo trials COMPLETE)."""
    values, best = [], -np.inf
    for t in sorted(study.trials, key=lambda x: x.number):
        if t.state == optuna.trial.TrialState.COMPLETE:
            best = max(best, t.value)
        values.append(best if best > -np.inf else np.nan)
    return values

evo_random = best_so_far(study_random)
evo_bayes  = best_so_far(study_bayes)
trials_x   = list(range(1, N_TRIALS + 1))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Optimización de Hiperparámetros — Evolución y Top-k', fontsize=13)

# Gráfico 1: curva de evolución
ax = axes[0]
ax.plot(trials_x, evo_random, color=COLORS[0], linewidth=2, label='Random Search')
ax.plot(trials_x, evo_bayes,  color=COLORS[1], linewidth=2, label='Bayesian TPE')
ax.set_xlabel('Trial #')
ax.set_ylabel('Mejor F1 acumulado')
ax.set_title('Evolución del mejor F1 por trial')
ax.legend(); ax.grid(True, alpha=0.4)
ax.set_xlim(1, N_TRIALS)

# Gráfico 2: distribución top-10
ax = axes[1]
top_r = df_top_random['f1_score'].values
top_b = df_top_bayes['f1_score'].values
x_pos = np.arange(K)
width = 0.35
ax.bar(x_pos - width/2, top_r, width, color=COLORS[0], alpha=0.8, label='Random Search')
ax.bar(x_pos + width/2, top_b, width, color=COLORS[1], alpha=0.8, label='Bayesian TPE')
ax.set_xlabel('Rank')
ax.set_ylabel('F1-score (weighted)')
ax.set_title(f'Top-{K} configuraciones por estrategia')
ax.set_xticks(x_pos); ax.set_xticklabels([f'#{i+1}' for i in range(K)])
ax.legend(); ax.grid(True, alpha=0.4, axis='y')

plt.tight_layout()
plt.savefig(LOGS_DIR / 'hpo_evolucion_topk.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Gráfico guardado: logs/hpo_evolucion_topk.png')

---
## 5. Resumen del espacio, presupuesto y decisión (config ganadora)

Síntesis de todo el experimento: espacio explorado, presupuesto utilizado y la configuración que se adopta para el modelo final.

In [ ]:
winner_f1   = winner_study.best_value
loser_study = study_random if winner_study is study_bayes else study_bayes
loser_name  = 'RandomSearch' if winner_name == 'BayesianTPE' else 'BayesianTPE'
loser_f1    = loser_study.best_value

# Tabla comparativa de estrategias
df_summary = pd.DataFrame([
    {
        'Estrategia':   'Random Search',
        'Trials total': N_TRIALS,
        'Completados':  len([t for t in study_random.trials if t.state == optuna.trial.TrialState.COMPLETE]),
        'Podados':      len([t for t in study_random.trials if t.state == optuna.trial.TrialState.PRUNED]),
        'Mejor F1':     round(study_random.best_value, 4),
    },
    {
        'Estrategia':   'Bayesian TPE',
        'Trials total': N_TRIALS,
        'Completados':  len([t for t in study_bayes.trials if t.state == optuna.trial.TrialState.COMPLETE]),
        'Podados':      len([t for t in study_bayes.trials if t.state == optuna.trial.TrialState.PRUNED]),
        'Mejor F1':     round(study_bayes.best_value, 4),
    },
])
print('\n=== Comparación de estrategias ===')
print(df_summary.to_string(index=False))

# Config ganadora
print(f'\n=== CONFIG GANADORA ({winner_name}) ===')
for param, val in bp.items():
    print(f'  {param:<22}: {val}')
print(f'  {"F1 (CV-5 medio)":<22}: {winner_f1:.4f}')

# Decisión final en JSON
decision = {
    'fecha':            datetime.now().strftime('%Y-%m-%d %H:%M'),
    'notebook':         '07_hpo_experiments',
    'espacio_busqueda': {k: list(v) if isinstance(v, list) else str(v) for k, v in SEARCH_SPACE.items()},
    'presupuesto': {
        'trials_por_estrategia': N_TRIALS,
        'cv_splits':             CV_SPLITS,
        'total_fits':            N_TRIALS * 2 * CV_SPLITS,
    },
    'ganadora': {
        'estrategia':   winner_name,
        'f1':           round(winner_f1, 6),
        'params':       bp,
        'artefacto':    'artifacts/hpo_best_model.pkl',
    },
    'comparacion': {
        winner_name: round(winner_f1, 4),
        loser_name:  round(loser_f1,  4),
        'diferencia': round(winner_f1 - loser_f1, 4),
    },
    'justificacion': (
        f'{winner_name} obtuvo F1={winner_f1:.4f} vs {loser_name} F1={loser_f1:.4f}. '
        f'Se selecciona {winner_name} como estrategia ganadora y su mejor configuración '
        f'como punto de partida para las siguientes etapas.'
    ),
}

with open(LOGS_DIR / 'hpo_decision.json', 'w', encoding='utf-8') as f:
    json.dump(decision, f, indent=2, ensure_ascii=False)
print('\n✅ Decisión guardada: logs/hpo_decision.json')

In [ ]:
# Resumen visual final
fig, ax = plt.subplots(figsize=(8, 4))
strategies = ['Random Search', 'Bayesian TPE']
f1_vals    = [study_random.best_value, study_bayes.best_value]
bar_colors = [COLORS[0], COLORS[1]]
bars = ax.bar(strategies, f1_vals, color=bar_colors, edgecolor='white', width=0.4)
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.002,
            f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Mejor F1 (CV-5)')
ax.set_title('Resultado final — Random Search vs Bayesian TPE', fontsize=12)
ax.set_ylim(0, min(1.0, max(f1_vals) * 1.15))
ax.grid(True, alpha=0.4, axis='y')
plt.tight_layout()
plt.savefig(LOGS_DIR / 'hpo_decision_final.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Gráfico de decisión: logs/hpo_decision_final.png')

---
## ✅ Resumen del notebook

| Punto | Qué se hizo | Archivos generados |
|---|---|---|
| **1. Experimentos comparables** | Random Search vs Bayesian TPE, mismo espacio y presupuesto, con pruning (MedianPruner) | — |
| **2. Logs + artefactos** | Todos los trials en CSV/JSON; modelo ganador serializado | `logs/hpo_all_trials.csv`, `logs/hpo_log.json`, `artifacts/hpo_best_model.pkl` |
| **3. Tabla top-k y gráfico** | Top-10 por estrategia + curva de evolución del mejor F1 | `logs/hpo_top10_*.csv`, `logs/hpo_evolucion_topk.png` |
| **4. Resumen y decisión** | Comparación final, espacio de búsqueda, presupuesto y config ganadora justificada | `logs/hpo_decision.json`, `logs/hpo_decision_final.png` |

### Archivos generados en `logs/`
- `hpo_all_trials.csv` — todos los trials de ambas estrategias
- `hpo_log.json` — log detallado por estrategia
- `hpo_top10_random.csv` / `hpo_top10_bayes.csv` — tablas top-k
- `hpo_evolucion_topk.png` — evolución + distribución top-k
- `hpo_decision.json` — decisión final documentada
- `hpo_decision_final.png` — comparación visual

### Artefactos en `artifacts/`
- `hpo_best_model.pkl` — vectorizador + clasificador con la config ganadora